# 07 — Cartographie des routes ferroviaires
**Projet** : ObRail MSPR 2025-2026  
**Auteure** : Jeannette  

**Objectif** : Visualiser géographiquement le taux de sous-desserte ferroviaire par pays européen sous forme de carte choropleth.

**Note** : le dataset ne contient pas de coordonnées GPS des gares. On représente donc le taux de sous-desserte au niveau du pays (carte choropleth) plutôt que par gare individuelle.

**Entrée** : `data/processed/routes_processed.csv`  
**Sortie** : `evaluation/plots/cartographie_sous_desserte.png`

## 0. Imports et configuration

In [2]:
from pathlib import Path

import pandas as pd
import plotly.express as px

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for folder in [cwd] + list(cwd.parents):
        if (folder / 'data').exists() and (folder / 'src').exists() and (folder / 'models').exists():
            return folder
    raise FileNotFoundError('Racine du projet introuvable.')

ROOT          = find_project_root()
PROCESSED_DIR = ROOT / 'data' / 'processed'
PLOT_DIR      = ROOT / 'evaluation' / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

Imports OK


## 1. Chargement des données

In [3]:
df = pd.read_csv(PROCESSED_DIR / 'routes_processed.csv', dtype={'days_of_week': str})
print(f'Shape : {df.shape}')
df[['departure_country', 'is_underserved']].head()

Shape : (25200, 21)


,departure_country,is_underserved
0,AT,0
1,AT,1
2,AT,0
3,AT,0
4,AT,0


## 2. Agrégation par pays de départ

Pour chaque pays, on calcule le nombre total de routes et le taux de sous-desserte (moyenne de `is_underserved`).

In [4]:
stats_pays = (
    df.groupby('departure_country')['is_underserved']
    .agg(['count', 'mean'])
    .rename(columns={'count': 'nb_routes', 'mean': 'taux_sous_desserte'})
    .reset_index()
)
stats_pays['taux_pct'] = (stats_pays['taux_sous_desserte'] * 100).round(1)
stats_pays = stats_pays.sort_values('taux_pct', ascending=False)

print('Taux de sous-desserte par pays :')
display(stats_pays)

Taux de sous-desserte par pays :


,departure_country,nb_routes,taux_sous_desserte,taux_pct
0,AL,3,1.000000,100.0
22,MK,1,1.000000,100.0
20,MD,2,1.000000,100.0
32,TR,16,0.937500,93.8
33,UA,58,0.931034,93.1
15,HU,34,0.882353,88.2
2,BE,12,0.833333,83.3
4,CH,22,0.636364,63.6
3,BG,22,0.590909,59.1
10,FI,11,0.545455,54.5


## 3. Conversion des codes pays ISO-2 vers ISO-3

Plotly a besoin de codes pays à 3 lettres (ISO-3) pour la cartographie. Nos données utilisent des codes à 2 lettres (ISO-2).

In [5]:
# Table de correspondance ISO-2 -> ISO-3 pour les pays présents
iso2_to_iso3 = {
    'FR': 'FRA', 'DE': 'DEU', 'DK': 'DNK', 'GB': 'GBR', 'IT': 'ITA',
    'ES': 'ESP', 'PL': 'POL', 'NL': 'NLD', 'BE': 'BEL', 'AT': 'AUT',
    'SE': 'SWE', 'CH': 'CHE', 'CZ': 'CZE', 'HU': 'HUN', 'SK': 'SVK',
    'PT': 'PRT', 'GR': 'GRC', 'RO': 'ROU', 'HR': 'HRV', 'SI': 'SVN',
    'FI': 'FIN', 'NO': 'NOR', 'IE': 'IRL', 'LU': 'LUX', 'LT': 'LTU',
    'EE': 'EST', 'LV': 'LVA', 'BG': 'BGR', 'RS': 'SRB', 'ME': 'MNE',
    'MK': 'MKD', 'AL': 'ALB', 'UA': 'UKR', 'MD': 'MDA', 'TR': 'TUR',
}

stats_pays['iso3'] = stats_pays['departure_country'].map(iso2_to_iso3)

# Vérifier qu'aucun pays n'a été oublié
manquants = stats_pays[stats_pays['iso3'].isna()]
if len(manquants) > 0:
    print('Pays sans correspondance ISO-3 :')
    print(manquants['departure_country'].tolist())
else:
    print('Tous les pays ont une correspondance ISO-3.')

Tous les pays ont une correspondance ISO-3.


## 4. Carte choropleth — taux de sous-desserte par pays

In [7]:
fig = px.choropleth(
    stats_pays,
    locations='iso3',
    color='taux_pct',
    hover_name='departure_country',
    hover_data={'nb_routes': True, 'taux_pct': True, 'iso3': False},
    color_continuous_scale='Reds',
    scope='europe',
    labels={'taux_pct': 'Taux de sous-desserte (%)'},
    title='Taux de sous-desserte ferroviaire par pays de départ',
)

fig.update_layout(
    title_x=0.5,
    geo=dict(showframe=False, showcoastlines=True),
    coloraxis_colorbar=dict(title='% sous-desservi'),
    margin=dict(l=0, r=0, t=50, b=0),
)

fig.show()

## 5. Sauvegarde de la carte

Note : l'export en PNG nécessite le package `kaleido` (`pip install kaleido`). Si l'export échoue, la carte reste visible dans le notebook.

In [8]:
try:
    fig.write_image(str(PLOT_DIR / 'cartographie_sous_desserte.png'), width=900, height=700, scale=2)
    print('Carte sauvegardee -> evaluation/plots/cartographie_sous_desserte.png')
except Exception as e:
    print('Export PNG echoue (installer kaleido : pip install kaleido)')
    print('La carte reste visible dans le notebook.')
    print(f'Detail : {e}')

# Sauvegarde alternative en HTML interactif (toujours disponible)
fig.write_html(str(PLOT_DIR / 'cartographie_sous_desserte.html'))
print('Carte interactive sauvegardee -> evaluation/plots/cartographie_sous_desserte.html')

Export PNG echoue (installer kaleido : pip install kaleido)
La carte reste visible dans le notebook.
Detail : 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

Carte interactive sauvegardee -> evaluation/plots/cartographie_sous_desserte.html


## 6. Carte complémentaire — volume de routes par pays

Pour contextualiser : un pays peut avoir un taux élevé mais peu de routes. Cette seconde carte montre le volume.

In [9]:
fig2 = px.choropleth(
    stats_pays,
    locations='iso3',
    color='nb_routes',
    hover_name='departure_country',
    color_continuous_scale='Blues',
    scope='europe',
    labels={'nb_routes': 'Nombre de routes'},
    title='Volume de routes ferroviaires par pays de départ',
)

fig2.update_layout(
    title_x=0.5,
    geo=dict(showframe=False, showcoastlines=True),
    margin=dict(l=0, r=0, t=50, b=0),
)

fig2.show()

fig2.write_html(str(PLOT_DIR / 'cartographie_volume_routes.html'))
print('Carte volume sauvegardee -> evaluation/plots/cartographie_volume_routes.html')

Carte volume sauvegardee -> evaluation/plots/cartographie_volume_routes.html


## 7. Synthèse

**Ce que montre la cartographie**
- Le taux de sous-desserte varie fortement d'un pays à l'autre, allant de moins de 2 % à plus de 90 % selon le pays de départ.
- Les taux les plus élevés concernent la Turquie (93,8 % sur 16 routes), l'Ukraine (93,1 % sur 58 routes), la Hongrie (88,2 % sur 34 routes) et la Belgique (83,3 % sur 12 routes). L'Albanie, la Macédoine du Nord et la Moldavie affichent un taux de 100 %, mais sur un échantillon de seulement 1 à 3 routes chacun : ce résultat n'est pas statistiquement représentatif et doit être interprété avec prudence.
- Les taux les plus faibles sont observés en Slovénie (1,7 %), Grèce (2,7 %), Lituanie (3,8 %), Portugal (4,1 %) et Roumanie (4,8 %), tous appuyés sur des échantillons de plus de 35 routes, ce qui renforce la fiabilité de ce constat.
Les pays disposant des plus gros volumes de routes du dataset — France (6 960 routes, 22,3 %), Espagne (4 789 routes, 20,1 %) et Allemagne (4 429 routes, 22,5 %) — présentent des taux de sous-desserte modérés et cohérents entre eux, malgré leurs réseaux ferroviaires très différents.
- Les Pays-Bas (37,6 % sur 567 routes) et l'Irlande (27,7 % sur 242 routes) se situent à un niveau intermédiaire, supérieur aux grands pays d'Europe occidentale mais nettement inférieur aux pays en tête du classement.

**Limite**
- En l'absence de coordonnées GPS des gares, la cartographie est faite au niveau du pays de départ et non par gare individuelle. Une version future avec les coordonnées GTFS permettrait une carte plus précise au niveau des liaisons.
Plusieurs pays (Albanie, Macédoine du Nord, Moldavie, Monténégro) reposent sur moins de 5 routes dans le dataset : leur taux de sous-desserte à 100 % reflète un effet d'échantillon réduit plutôt qu'une tendance structurelle avérée. Une lecture par volume de routes (carte bonus) doit systématiquement accompagner la carte de taux pour éviter toute conclusion hâtive sur ces pays.

**Fichiers produits**
- `evaluation/plots/cartographie_sous_desserte.png` (et .html)
- `evaluation/plots/cartographie_volume_routes.html`